# EKF: diagrama de bloques — versión 2

Resumen visual del **EKF discreto de primer orden con ruido aditivo**. Esta versión monocromática presenta una secuencia explícita: tres pasos de predicción y siete pasos de actualización. Incluye además una figura compacta y formal para publicación. Conserva el ciclo, los supuestos, la notación y la interpretación de las matrices de la figura original. Los archivos de la primera versión se mantienen intactos.

El estado se estima mediante una distribución gaussiana aproximada, representada por el punto $\hat{\mathbf{x}}$ y la nube $\mathbf{P}$. La media se propaga con los modelos no lineales $\mathbf{f},\mathbf{h}$; el error se aproxima localmente con sus Jacobianos $\mathbf{F},\mathbf{H}$.

**Ejecución:** activar Conda `fermentation`, seleccionar el kernel `Python (fermentation)` y ejecutar desde esta carpeta. El notebook genera `ekf_general_block_diagram_v2.png` y `.svg` (guía detallada), y `ekf_general_block_diagram_paper.png` y `.svg` (figura compacta). Los SVG son vectoriales y mantienen el texto editable. Las covarianzas $\mathbf{Q}_k,\mathbf{R}_{k+1}$ pueden ser constantes o variables; no son realizaciones del ruido. Los ruidos se asumen blancos, de media cero, gaussianos, independientes entre sí y del error inicial.

La inicialización con superíndice $+$ designa la creencia disponible en $t_0$. Si existe una medición inicial que aún no se incorporó, actualizar primero esa creencia. En la rama sin medición, $+$ designa la salida del intervalo y coincide con $-$.


Incluye una tercera figura simplificada con decisión explícita de disponibilidad de medición, exportada como `ekf_general_block_diagram_paper_with_measurement.png` y `.svg`.


In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'fermentation_model').exists())
DIAGRAM_DIR = ROOT / 'fermentation_model/laboratory_2026/notebooks/EKF/diagrams'
DIAGRAM_DIR.mkdir(exist_ok=True)


## Orden de ejecución

La predicción se enumera como una secuencia de implementación: **(1)** predecir el estado, **(2)** evaluar el Jacobiano $\mathbf{F}_k$ en la estimación anterior y las entradas conocidas, **(3)** predecir la covarianza. El tercer cálculo depende del segundo. Los dos primeros usan la misma creencia anterior y pueden intercambiarse; la figura no exige ejecución paralela. Antes de corregir deben estar disponibles tanto $\hat{\mathbf{x}}_{k+1}^{-}$ como $\mathbf{P}_{k+1}^{-}$.

En un modelo continuo, el estado y las sensibilidades pueden integrarse conjuntamente. La enumeración describe las dependencias del EKF discreto; no impone que el integrador deba realizar tres llamadas separadas.


In [ ]:
from pathlib import Path
import sys
from IPython import get_ipython

if Path(sys.prefix).name.lower() != 'fermentation':
    raise RuntimeError('Ejecutar este notebook con el kernel fermentation, desde un entorno Conda activado.')

_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic('matplotlib', 'inline')

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Polygon

mpl.rcParams.update({'font.family': 'DejaVu Sans', 'mathtext.fontset': 'dejavusans', 'svg.fonttype': 'none'})

INK, MUTED = '#111111', '#444444'
BLUE, BLUE_BG = '#111111', '#ffffff'
GREEN, GREEN_BG = '#111111', '#ffffff'
GOLD, GOLD_BG = '#111111', '#ffffff'
GRAY, GRAY_BG = '#111111', '#ffffff'
STAR = '#111111'
fig, ax = plt.subplots(figsize=(18, 20.5))
fig.subplots_adjust(left=0.025, right=0.985, bottom=0.018, top=0.985)
ax.set(xlim=(0, 180), ylim=(0, 205))
ax.axis('off')
labels = []

def box(x0, y0, x1, y1, fill=GRAY_BG, edge=GRAY, lw=1.5):
    ax.add_patch(FancyBboxPatch((x0, y0), x1-x0, y1-y0,
        boxstyle='round,pad=0,rounding_size=1.4', facecolor=fill,
        edgecolor=edge, linewidth=lw, zorder=2))

def text(x, y, value, size=12, color=INK, ha='left', weight='normal', style='normal', bounds=None):
    artist = ax.text(x, y, value, fontsize=size, color=color, ha=ha, va='center',
                     fontweight=weight, fontstyle=style, zorder=5)
    if bounds is not None:
        labels.append((artist, bounds))
    return artist

def arrow(a, b, color=INK, lw=1.6):
    ax.add_patch(FancyArrowPatch(a, b, arrowstyle='-|>', mutation_scale=15,
        color=color, linewidth=lw, shrinkA=0, shrinkB=0, zorder=3))

def route(points, color=INK):
    ax.plot(*zip(*points), color=color, linewidth=1.6, zorder=1)
    arrow(points[-2], points[-1], color)

def line(y, x0=16.5, x1=108.5, color='#b0b0b0'):
    ax.plot([x0, x1], [y, y], color=color, linewidth=0.9, zorder=3)

# Título y alcance: la figura describe una variante bien definida del EKF.
text(90, 200, 'Filtro de Kalman extendido · guía de memoria', size=23, ha='center', weight='bold')
text(90, 194.7, 'Versión 2   |   secuencia de cálculo · tiempo discreto · ruido aditivo',
     size=13, color=MUTED, ha='center')

# Modelo generador: las realizaciones de x, w, v son desconocidas.
box(14, 168, 111, 188, '#ffffff', GRAY)
text(17, 184.7, 'MODELO Y SUPUESTOS', size=13, weight='bold', bounds=(14,168,111,188))
text(17, 179.8, r'$\mathbf{x}_{k+1}=\mathbf{f}(\mathbf{x}_k,\mathbf{u}_k)+\mathbf{w}_k$',
     size=15, bounds=(14,168,111,188))
text(63, 179.8, r'$\mathbf{z}_{k+1}=\mathbf{h}(\mathbf{x}_{k+1})+\mathbf{v}_{k+1}$',
     size=15, bounds=(14,168,111,188))
text(17, 174.8, r'$\mathbf{w}_k\sim\mathcal{N}(\mathbf{0},\mathbf{Q}_k)$', size=13,
     bounds=(14,168,111,188))
text(63, 174.8, r'$\mathbf{v}_{k+1}\sim\mathcal{N}(\mathbf{0},\mathbf{R}_{k+1})$', size=13,
     bounds=(14,168,111,188))
text(17, 170.6, 'Ruidos blancos, independientes entre sí y del error inicial.',
     size=11.5, color=MUTED, bounds=(14,168,111,188))
arrow((62.5,168),(62.5,164))

# Inicialización y creencia disponible al comenzar un intervalo.
box(14,150,111,164,GOLD_BG,GOLD)
text(17,160.5,'0 · INICIALIZAR UNA VEZ',size=13,color=GOLD,weight='bold',bounds=(14,150,111,164))
text(17,155.7,r'$\hat{\mathbf{x}}_0^+$: estimación inicial',size=12.8,bounds=(14,150,111,164))
text(65,155.7,r'$\mathbf{P}_0^+$: covarianza inicial del error',size=12,bounds=(14,150,111,164))
text(17,152,'Especificar los modelos f, h y las covarianzas Q, R (constantes o variables).',
     size=11.3,color=MUTED,bounds=(14,150,111,164))
arrow((62.5,150),(62.5,146))
box(30,137,95,146)
text(62.5,142.5,r'$\hat{\mathbf{x}}_k^+,\ \mathbf{P}_k^+$',size=16,ha='center',bounds=(30,137,95,146))
text(62.5,138.9,r'Creencia disponible en $t_k$',size=11,color=MUTED,ha='center',bounds=(30,137,95,146))
# Predicción secuencial: tres cálculos dentro de una sola etapa.
arrow((62.5,137),(62.5,130))
box(14,106,111,130)
text(17,126.7,'1 · PREDICCIÓN',size=13,weight='bold',bounds=(14,106,111,130))
text(109,126.7,r'Entradas: $\mathbf{u}_k,\ \mathbf{Q}_k$',size=11.5,ha='right',bounds=(14,106,111,130))
text(18,121.5,r'①  $\hat{\mathbf{x}}_{k+1}^{-}=\mathbf{f}(\hat{\mathbf{x}}_k^+,\mathbf{u}_k)$',
     size=15,bounds=(14,106,111,130))
text(70,121.5,'Predecir el estado.',size=11.5,bounds=(14,106,111,130))
text(18,115.1,r'②  $\mathbf{F}_k=\left.\frac{\partial\mathbf{f}}{\partial\mathbf{x}}\right|_{\hat{\mathbf{x}}_k^+,\mathbf{u}_k}$',
     size=15,bounds=(14,106,111,130))
text(16.8,115.1,'★',size=13,ha='right')
text(70,115.1,'Evaluar el Jacobiano de la transición.',size=11.5,bounds=(14,106,111,130))
text(18,109.4,r'③  $\mathbf{P}_{k+1}^{-}=\mathbf{F}_k\mathbf{P}_k^+\mathbf{F}_k^\top+\mathbf{Q}_k$',
     size=15,bounds=(14,106,111,130))
text(70,109.4,'Predecir la covarianza del error.',size=11.5,bounds=(14,106,111,130))
arrow((62.5,106),(62.5,103))
box(36,95,89,103)
text(62.5,100.1,r'$\hat{\mathbf{x}}_{k+1}^-,\ \mathbf{P}_{k+1}^-$',size=15,ha='center',bounds=(36,95,89,103))
text(62.5,96.7,'A priori · antes de usar la nueva medición',size=10.6,color=MUTED,ha='center',bounds=(36,95,89,103))
arrow((62.5,95),(62.5,92))

# Bifurcación explícita; la ruta sin medición también produce una salida válida.
ax.add_patch(Polygon([(49,86),(62.5,92),(76,86),(62.5,80)],closed=True,
    facecolor=GOLD_BG,edgecolor=GOLD,linewidth=1.5,zorder=2))
text(62.5,88,'¿Hay medición',size=11.5,ha='center')
text(62.5,84.6,r'en $t_{k+1}$?',size=11.5,ha='center')
arrow((62.5,80),(62.5,76),GREEN)
text(65,78.2,'SÍ',size=11,color=GREEN,weight='bold')
route([(76,86),(125,86),(125,17.2),(102,17.2)])
text(80,88.3,'NO',size=11,weight='bold')
box(90,80,125,93,'white',GRAY,lw=1)
arrow((76,86),(90,86))
text(107.5,90.2,'SIN MEDICIÓN',size=10,ha='center',weight='bold',bounds=(90,80,125,93))
text(107.5,86.3,r'$\hat{\mathbf{x}}_{k+1}^{+}=\hat{\mathbf{x}}_{k+1}^{-}$',size=12,ha='center',bounds=(90,80,125,93))
text(107.5,82.3,r'$\mathbf{P}_{k+1}^{+}=\mathbf{P}_{k+1}^{-}$',size=12,ha='center',bounds=(90,80,125,93))

# Actualización en tres grupos, manteniendo los siete pasos del algoritmo.
box(14,26,111,76,GREEN_BG,GREEN)
text(17,72.5,'2 · ACTUALIZACIÓN CON LA MEDICIÓN',size=13,color=GREEN,weight='bold',bounds=(14,26,111,76))
text(109,78.3,r'Entradas: $\mathbf{z}_{k+1},\ \mathbf{R}_{k+1}$',size=11.5,color=GREEN,ha='right')
arrow((104,77.2),(104,76),GREEN)
text(17,67.7,'A · Comparar lo medido con lo esperado',size=10.8,color=GREEN,weight='bold',bounds=(14,26,111,76))
equations = [
    (63.7,r'①  $\hat{\mathbf{z}}_{k+1}=\mathbf{h}(\hat{\mathbf{x}}_{k+1}^{-})$',14),
    (59.5,r'②  $\mathbf{H}_{k+1}=\left.\frac{\partial\mathbf{h}}{\partial\mathbf{x}}\right|_{\hat{\mathbf{x}}_{k+1}^{-}}$',14),
    (55.2,r'③  $\mathbf{y}_{k+1}=\mathbf{z}_{k+1}-\hat{\mathbf{z}}_{k+1}$   · innovación',14),
    (46.5,r'④  $\mathbf{S}_{k+1}=\mathbf{H}_{k+1}\mathbf{P}_{k+1}^{-}\mathbf{H}_{k+1}^{\top}+\mathbf{R}_{k+1}$',14),
    (42.3,r'⑤  $\mathbf{K}_{k+1}=\mathbf{P}_{k+1}^{-}\mathbf{H}_{k+1}^{\top}\mathbf{S}_{k+1}^{-1}$',14),
    (33.7,r'⑥  $\hat{\mathbf{x}}_{k+1}^{+}=\hat{\mathbf{x}}_{k+1}^{-}+\mathbf{K}_{k+1}\mathbf{y}_{k+1}$',14),
    (29.0,r'⑦  $\mathbf{P}_{k+1}^{+}=(\mathbf{I}-\mathbf{K}_{k+1}\mathbf{H}_{k+1})\mathbf{P}_{k+1}^{-}(\mathbf{I}-\mathbf{K}_{k+1}\mathbf{H}_{k+1})^{\top}+\mathbf{K}_{k+1}\mathbf{R}_{k+1}\mathbf{K}_{k+1}^{\top}$',12),
]
for yy, eq, size in equations:
    text(18,yy,eq,size=size,bounds=(14,26,111,76))
text(16.8,59.5,'★',size=13,color=STAR,ha='right')
line(52.2)
text(17,50.0,'B · S: covarianza de la innovación  /  K: ganancia de Kalman',
     size=10.8,color=GREEN,weight='bold',bounds=(14,26,111,76))
line(39.5)
text(17,37.3,'C · Corregir el estado y su covarianza (forma de Joseph)',
     size=10.8,color=GREEN,weight='bold',bounds=(14,26,111,76))
arrow((62.5,26),(62.5,22),GREEN)
box(24,12,102,22)
text(63,18.1,r'$\hat{\mathbf{x}}_{k+1}^+,\ \mathbf{P}_{k+1}^+$',size=16,ha='center',bounds=(24,12,102,22))
text(63,14.2,'Salida del filtro · actualizada si hubo medición',size=11,color=MUTED,ha='center',bounds=(24,12,102,22))
route([(24,17.2),(6,17.2),(6,142),(30,142)],BLUE)
text(14,8.1,r'Siguiente intervalo: $k\leftarrow k+1$',size=12,color=BLUE,weight='bold')

# Apoyos laterales: notación, significado estadístico, discretización y dimensiones.
box(128,160,177,188)
text(131,184.7,'CÓMO LEER LA FIGURA',size=12.5,weight='bold',bounds=(128,160,177,188))
side = [
    (179.9,r'$\hat{\mathbf{x}}$: estimación del estado'),
    (175.5,r'$-$: antes de incorporar la medición'),
    (171.1,r'$+$: después de incorporarla / salida'),
    (166.7,'★ Jacobiano: linealización local'),
    (162.5,'I: identidad · f, h pueden depender de t y u.'),
]
for yy,value in side:
    text(131,yy,value,size=11.2,color=MUTED,bounds=(128,160,177,188))
box(128,117,177,156)
text(131,152.5,'QUÉ SIGNIFICAN LAS COVARIANZAS',size=11.8,weight='bold',bounds=(128,117,177,156))
text(131,147.7,'P: covarianza aproximada del error.',size=11.5,bounds=(128,117,177,156))
text(131,143.2,r'$P_{ii}=\sigma_i^2$; $P_{ij}$: covarianza cruzada.',size=11.7,bounds=(128,117,177,156))
text(131,138.6,'Q: covarianza de proceso del intervalo.',size=11.5,bounds=(128,117,177,156))
text(131,134.2,'R: covarianza del ruido de medición.',size=11.5,bounds=(128,117,177,156))
text(131,129.7,r'$\mathbf{P}^{+}=\mathbf{P}^{-}-\mathbf{K}\mathbf{S}\mathbf{K}^{\top}\preceq\mathbf{P}^{-}$',size=14,bounds=(128,117,177,156))
text(131,124.7,'La actualización reduce la covarianza estimada.',size=10.8,bounds=(128,117,177,156))
text(131,120.5,'No garantiza reducir el error real en cada paso.',size=10.8,color=MUTED,bounds=(128,117,177,156))
box(128,73,177,113,'#ffffff',GOLD)
text(131,109.5,'SI TU MODELO ES CONTINUO',size=12.5,color=GOLD,weight='bold',bounds=(128,73,177,113))
text(131,104.8,r'$\dot{\mathbf{x}}=\mathbf{g}(\mathbf{x},\mathbf{u})$',size=15,bounds=(128,73,177,113))
text(131,100.1,'f es el mapa obtenido al avanzar el intervalo.',size=10.8,bounds=(128,73,177,113))
text(131,95.5,r'Ejemplo Euler: $\mathbf{f}=\mathbf{x}+\Delta t_k\,\mathbf{g}$',size=12.2,bounds=(128,73,177,113))
text(131,89.8,r'$\mathbf{F}_k=\mathbf{I}+\Delta t_k\left.\frac{\partial\mathbf{g}}{\partial\mathbf{x}}\right|_{\hat{\mathbf{x}}_k^+,\mathbf{u}_k}$',size=14,bounds=(128,73,177,113))
text(131,84.1,'F es el Jacobiano de f, no directamente de g.',size=10.7,weight='bold',bounds=(128,73,177,113))
text(131,79.5,'Q debe corresponder al intervalo y puede variar.',size=10.7,bounds=(128,73,177,113))
text(131,75.4,'Con otros integradores, derivar el mapa usado.',size=10.7,color=MUTED,bounds=(128,73,177,113))
box(128,12,177,69)
text(131,65.5,'MAPA DE DIMENSIONES Y ROLES',size=11.8,weight='bold',bounds=(128,12,177,69))
text(131,60.9,'n: estados  ·  m: mediciones  ·  p: entradas',size=11,color=MUTED,bounds=(128,12,177,69))
rows = [
    (55.8,r'$\hat{\mathbf{x}}$',r'$n\times1$','estado estimado'),
    (50.6,r'$\mathbf{P},\mathbf{Q},\mathbf{F},\mathbf{I}$',r'$n\times n$','espacio del estado'),
    (45.4,r'$\mathbf{z},\hat{\mathbf{z}},\mathbf{y}$',r'$m\times1$','espacio de medición'),
    (40.2,r'$\mathbf{R},\mathbf{S}$',r'$m\times m$','ruido / innovación'),
    (35.0,r'$\mathbf{H}$',r'$m\times n$','estado → medición'),
    (29.8,r'$\mathbf{K}$',r'$n\times m$','innovación → estado'),
    (24.6,r'$\mathbf{u}$',r'$p\times1$','entrada conocida'),
]
for yy,symbol,dim,role in rows:
    text(131,yy,symbol,size=12.5,bounds=(128,12,177,69))
    text(146,yy,dim,size=11.5,bounds=(128,12,177,69))
    text(156,yy,role,size=10.3,color=MUTED,bounds=(128,12,177,69))
text(131,19.5,'f, h: modelos especificados; F, H: sus Jacobianos.',size=10.4,bounds=(128,12,177,69))
text(131,15.2,'x real, w y v: desconocidos; no se suministran.',size=10.4,color=MUTED,bounds=(128,12,177,69))
text(90,3.7,'La media usa las funciones no lineales; la covarianza usa sus Jacobianos locales.',
     size=12.5,color=MUTED,ha='center')

# Comprobación de maquetación: cada texto debe caber en su recuadro.
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
overflows = []
for artist, bounds in labels:
    x0,y0,x1,y1 = bounds
    a = ax.transData.transform((x0,y0))
    b = ax.transData.transform((x1,y1))
    extent = artist.get_window_extent(renderer)
    if extent.x0 < a[0]-1 or extent.x1 > b[0]+1 or extent.y0 < a[1]-1 or extent.y1 > b[1]+1:
        overflows.append(artist.get_text())
if overflows:
    raise RuntimeError('Texto fuera de recuadros: ' + repr(overflows))

output_path = (DIAGRAM_DIR / 'ekf_general_block_diagram_v2.png')
fig.savefig(output_path, dpi=180, facecolor='white')
fig.savefig(output_path.with_suffix('.svg'), facecolor='white')
plt.show()
print(f'Figura detallada guardada (PNG y SVG): {output_path.resolve()}')


## Figura simplificada para paper

Figura monocromática de dos bloques, con tipografía serif y ecuaciones numeradas. Conserva la evaluación de los Jacobianos y la dirección predicción → corrección → siguiente predicción. La corrección se realiza únicamente si llega una medición.

Para reducir la extensión, la predicción de la medición se incluye dentro de la innovación y se usa $\mathbf{P}^+=\mathbf{P}^- -\mathbf{K}\mathbf{S}\mathbf{K}^{\top}$, equivalente a Joseph con la ganancia indicada y aritmética exacta. La guía detallada mantiene Joseph para recordar su forma numéricamente robusta. En esta figura hay seis operaciones de corrección porque $\hat{\mathbf{z}}$ no se calcula en una línea independiente.

**Pie sugerido:** Esquema del filtro de Kalman extendido de primer orden para un sistema discreto con ruido aditivo independiente. La etapa de predicción propaga la estimación del estado y su covarianza; la corrección incorpora la nueva observación mediante la innovación y la ganancia de Kalman. Los superíndices $-$ y $+$ indican la creencia antes y después de la corrección, respectivamente. Si no hay observación, la predicción se conserva como salida del intervalo.


In [ ]:
# Figura compacta para publicación: tipografía serif, blanco y negro.
# Las ecuaciones son las mismas del EKF; se agrupan términos para abreviar.
with mpl.rc_context({'font.family': 'STIXGeneral', 'mathtext.fontset': 'stix',
                     'svg.fonttype': 'none', 'font.size': 11}):
    paper_fig, paper_ax = plt.subplots(figsize=(7.5, 4.6))
    paper_fig.subplots_adjust(left=0.015,right=0.985,bottom=0.025,top=0.98)
    paper_ax.set(xlim=(0,160),ylim=(0,100))
    paper_ax.axis('off')
    paper_texts = []

    def p_box(x0,y0,x1,y1):
        paper_ax.add_patch(mpl.patches.Rectangle((x0,y0),x1-x0,y1-y0,
            facecolor='white',edgecolor='black',linewidth=0.9,zorder=2))

    def p_text(x,y,s,size=11,ha='left',weight='normal',bounds=None):
        art = paper_ax.text(x,y,s,fontsize=size,ha=ha,va='center',
                            color='black',fontweight=weight,zorder=5)
        if bounds is not None:
            paper_texts.append((art,bounds))
        return art

    def p_arrow(a,b):
        paper_ax.add_patch(FancyArrowPatch(a,b,arrowstyle='-|>',
            mutation_scale=10,linewidth=0.9,color='black',shrinkA=0,shrinkB=0,zorder=3))

    # Creencia inicial y entrada al primer ciclo.
    p_box(8,85,70,97)
    p_text(39,91,r'Inicialización: $\hat{\mathbf{x}}_0^+,\ \mathbf{P}_0^+$',size=11,ha='center',bounds=(8,85,70,97))
    p_arrow((39,85),(39,76))

    # Tres operaciones de predicción, con F evaluada en el estado anterior.
    p_box(8,32,70,76)
    p_text(39,71.5,'Predicción',size=12,ha='center',weight='bold',bounds=(8,32,70,76))
    paper_ax.plot([8,70],[67.5,67.5],color='black',linewidth=0.65,zorder=3)
    p_text(12,63.5,r'Entradas: $\mathbf{u}_k,\ \mathbf{Q}_k$',size=10,bounds=(8,32,70,76))
    p_text(12,57,r'(1) $\hat{\mathbf{x}}_{k+1}^{-}=\mathbf{f}(\hat{\mathbf{x}}_k^+,\mathbf{u}_k)$',size=11.5,bounds=(8,32,70,76))
    p_text(12,47,r'(2) $\mathbf{F}_k=\left.\frac{\partial\mathbf{f}}{\partial\mathbf{x}}\right|_{\hat{\mathbf{x}}_k^+,\mathbf{u}_k}$',size=11.5,bounds=(8,32,70,76))
    p_text(12,37,r'(3) $\mathbf{P}_{k+1}^{-}=\mathbf{F}_k\mathbf{P}_k^+\mathbf{F}_k^\top+\mathbf{Q}_k$',size=11.5,bounds=(8,32,70,76))

    # Una única flecha hacia corrección; el lazo inferior regresa a predicción.
    p_arrow((70,54),(88,54))
    p_text(79,60,r'$\hat{\mathbf{x}}_{k+1}^-$',size=10,ha='center')
    p_text(79,48,r'$\mathbf{P}_{k+1}^-$',size=10,ha='center')
    p_text(121,95,r'$\mathbf{z}_{k+1},\ \mathbf{R}_{k+1}$',size=12,ha='center')
    p_arrow((121,90),(121,86))

    p_box(88,22,154,86)
    p_text(121,81.5,'Corrección',size=12,ha='center',weight='bold',bounds=(88,22,154,86))
    paper_ax.plot([88,154],[77.5,77.5],color='black',linewidth=0.65,zorder=3)
    compact = [
        (70,r'(1) $\mathbf{H}_{k+1}=\left.\frac{\partial\mathbf{h}}{\partial\mathbf{x}}\right|_{\hat{\mathbf{x}}_{k+1}^{-}}$'),
        (60.5,r'(2) $\mathbf{y}_{k+1}=\mathbf{z}_{k+1}-\mathbf{h}(\hat{\mathbf{x}}_{k+1}^-)$'),
        (52,r'(3) $\mathbf{S}_{k+1}=\mathbf{H}_{k+1}\mathbf{P}_{k+1}^{-}\mathbf{H}_{k+1}^\top+\mathbf{R}_{k+1}$'),
        (43.5,r'(4) $\mathbf{K}_{k+1}=\mathbf{P}_{k+1}^{-}\mathbf{H}_{k+1}^\top\mathbf{S}_{k+1}^{-1}$'),
        (35,r'(5) $\hat{\mathbf{x}}_{k+1}^{+}=\hat{\mathbf{x}}_{k+1}^{-}+\mathbf{K}_{k+1}\mathbf{y}_{k+1}$'),
        (26.5,r'(6) $\mathbf{P}_{k+1}^{+}=\mathbf{P}_{k+1}^{-}-\mathbf{K}_{k+1}\mathbf{S}_{k+1}\mathbf{K}_{k+1}^\top$'),
    ]
    for yy,eq in compact:
        p_text(92,yy,eq,size=10.7,bounds=(88,22,154,86))

    paper_ax.plot([121,121,39,39],[22,14,14,25],color='black',linewidth=0.9,zorder=1)
    p_arrow((39,25),(39,32))
    p_text(80,18,r'$k\leftarrow k+1$',size=10.5,ha='center')
    p_text(80,8.5,'EKF discreto de primer orden con ruido aditivo independiente.',size=10,ha='center')
    p_text(80,3,r'Sin medición: $\hat{\mathbf{x}}_{k+1}^{+}=\hat{\mathbf{x}}_{k+1}^{-}$, $\mathbf{P}_{k+1}^{+}=\mathbf{P}_{k+1}^{-}$.',size=10,ha='center')

    paper_fig.canvas.draw()
    pr = paper_fig.canvas.get_renderer()
    overflow = []
    for artist,bounds in paper_texts:
        x0,y0,x1,y1=bounds
        a=paper_ax.transData.transform((x0,y0))
        b=paper_ax.transData.transform((x1,y1))
        ext=artist.get_window_extent(pr)
        if ext.x0<a[0]-1 or ext.x1>b[0]+1 or ext.y0<a[1]-1 or ext.y1>b[1]+1:
            overflow.append(artist.get_text())
    if overflow:
        raise RuntimeError('Texto fuera del diagrama de paper: '+repr(overflow))
    paper_path=(DIAGRAM_DIR / 'ekf_general_block_diagram_paper.png')
    paper_fig.savefig(paper_path,dpi=300,facecolor='white')
    paper_fig.savefig(paper_path.with_suffix('.svg'),facecolor='white')
    plt.show()
    print(f'Figura de paper guardada (PNG y SVG): {paper_path.resolve()}')


## Figura simplificada con disponibilidad de medición

Versión adicional del diagrama de paper, monocromática y con una decisión explícita después de cada predicción: **¿hay una medición correspondiente al instante $t_{k+1}$?**

- **Sí:** se ejecutan los seis cálculos compactos de corrección, con la misma ganancia estándar y actualización de covarianza de la figura de paper anterior.
- **No:** la predicción del estado y su covarianza pasan a ser la salida del intervalo: $\hat{\mathbf{x}}_{k+1}^+=\hat{\mathbf{x}}_{k+1}^-$ y $\mathbf{P}_{k+1}^+=\mathbf{P}_{k+1}^-$.
- **Ambas ramas:** producen una creencia en $t_{k+1}$ y avanzan el índice $k\leftarrow k+1$. La inicialización no se repite.

El índice de $\mathbf{z}_{k+1}$ identifica el instante de la medición, no su número de llegada. Por ejemplo, si se hacen diez predicciones entre mediciones, se corrige en $k+10$ con $\mathbf{z}_{k+10}$ y la predicción de ese mismo instante. Las mediciones recibidas con retraso requieren un tratamiento específico; esta figura representa mediciones disponibles en el instante correspondiente.

El superíndice $+$ identifica la salida del intervalo; en la rama sin medición esa salida coincide con la predicción, aunque no se haya realizado una corrección. La expresión compacta de $\mathbf{P}^+$ es equivalente a Joseph con la ganancia indicada y aritmética exacta.

La celda siguiente se puede ejecutar de forma independiente en `fermentation`. Genera `ekf_general_block_diagram_paper_with_measurement.png` y `.svg`; las dos figuras anteriores se conservan.

**Pie sugerido:** Filtro de Kalman extendido de primer orden con disponibilidad intermitente de mediciones. La predicción se realiza en cada intervalo; la corrección se ejecuta únicamente cuando existe una observación del instante correspondiente. Si no hay observación, la predicción se conserva como salida y alimenta el intervalo siguiente.


In [ ]:
# Diagrama simplificado adicional: decisión explícita de disponibilidad de medición.
# Esta celda es independiente de las celdas que dibujan las figuras anteriores.
from pathlib import Path
import sys
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon, FancyArrowPatch

if Path(sys.prefix).name.lower() != 'fermentation':
    raise RuntimeError('Ejecutar con el kernel fermentation desde el entorno Conda activado.')

with mpl.rc_context({'font.family':'STIXGeneral','mathtext.fontset':'stix',
                     'svg.fonttype':'none','font.size':11}):
    decision_fig, decision_ax = plt.subplots(figsize=(8.4,6.7))
    decision_fig.subplots_adjust(left=0.02,right=0.985,bottom=0.018,top=0.985)
    decision_ax.set(xlim=(0,200),ylim=(-8,150))
    decision_ax.axis('off')
    decision_labels = []

    def d_box(x0,y0,x1,y1):
        decision_ax.add_patch(Rectangle((x0,y0),x1-x0,y1-y0,
            facecolor='white',edgecolor='black',linewidth=0.9,zorder=2))

    def d_text(x,y,s,size=11,ha='left',weight='normal',bounds=None):
        artist=decision_ax.text(x,y,s,fontsize=size,color='black',ha=ha,
            va='center',fontweight=weight,zorder=5)
        if bounds is not None:
            decision_labels.append((artist,bounds))
        return artist

    def d_arrow(a,b):
        decision_ax.add_patch(FancyArrowPatch(a,b,arrowstyle='-|>',
            mutation_scale=11,linewidth=0.9,color='black',shrinkA=0,shrinkB=0,zorder=3))

    def d_route(points):
        decision_ax.plot(*zip(*points),color='black',linewidth=0.9,zorder=1)
        d_arrow(points[-2],points[-1])

    # Inicialización: ocurre una sola vez, antes del ciclo.
    d_box(8,134,92,147)
    d_text(50,140.5,r'Inicialización: $\hat{\mathbf{x}}_0^+,\ \mathbf{P}_0^+$',
        size=12,ha='center',bounds=(8,134,92,147))
    d_arrow((50,134),(50,129))

    # La predicción siempre avanza un intervalo, con medición o sin ella.
    d_box(8,82,92,129)
    d_text(50,124.3,'Predicción',size=13,ha='center',weight='bold',bounds=(8,82,92,129))
    decision_ax.plot([8,92],[120,120],color='black',linewidth=0.65,zorder=3)
    d_text(12,115.5,r'Entradas: $\mathbf{u}_k,\ \mathbf{Q}_k$',size=10.5,bounds=(8,82,92,129))
    d_text(12,108,r'(1) $\hat{\mathbf{x}}_{k+1}^{-}=\mathbf{f}(\hat{\mathbf{x}}_k^+,\mathbf{u}_k)$',
        size=12,bounds=(8,82,92,129))
    d_text(12,98,r'(2) $\mathbf{F}_k=\left.\frac{\partial\mathbf{f}}{\partial\mathbf{x}}\right|_{\hat{\mathbf{x}}_k^+,\mathbf{u}_k}$',
        size=12,bounds=(8,82,92,129))
    d_text(12,87,r'(3) $\mathbf{P}_{k+1}^{-}=\mathbf{F}_k\mathbf{P}_k^+\mathbf{F}_k^\top+\mathbf{Q}_k$',
        size=12,bounds=(8,82,92,129))
    d_arrow((50,82),(50,76))

    # El índice de z identifica el instante de adquisición, no el número de datos.
    decision_ax.add_patch(Polygon([(26,66),(50,76),(74,66),(50,56)],closed=True,
        facecolor='white',edgecolor='black',linewidth=0.9,zorder=2))
    d_text(50,69,'¿Hay medición',size=11,ha='center')
    d_text(50,63,r'$\mathbf{z}_{k+1}$ en $t_{k+1}$?',size=11,ha='center')

    # Sí: efectuar la corrección. Las ecuaciones compactas coinciden con el paper original.
    d_arrow((74,66),(110,66))
    d_text(91,69.5,'Sí',size=11,ha='center')
    d_text(152,105,r'$\mathbf{z}_{k+1},\ \mathbf{R}_{k+1}$',size=12,ha='center')
    d_arrow((152,100),(152,95))
    d_box(110,32,194,95)
    d_text(152,90.5,'Corrección',size=13,ha='center',weight='bold',bounds=(110,32,194,95))
    decision_ax.plot([110,194],[86.5,86.5],color='black',linewidth=0.65,zorder=3)
    correction = [
        (80.5,r'(1) $\mathbf{H}_{k+1}=\left.\frac{\partial\mathbf{h}}{\partial\mathbf{x}}\right|_{\hat{\mathbf{x}}_{k+1}^{-}}$'),
        (71.5,r'(2) $\mathbf{y}_{k+1}=\mathbf{z}_{k+1}-\mathbf{h}(\hat{\mathbf{x}}_{k+1}^-)$'),
        (62.5,r'(3) $\mathbf{S}_{k+1}=\mathbf{H}_{k+1}\mathbf{P}_{k+1}^{-}\mathbf{H}_{k+1}^{\top}+\mathbf{R}_{k+1}$'),
        (53.5,r'(4) $\mathbf{K}_{k+1}=\mathbf{P}_{k+1}^{-}\mathbf{H}_{k+1}^{\top}\mathbf{S}_{k+1}^{-1}$'),
        (44.5,r'(5) $\hat{\mathbf{x}}_{k+1}^{+}=\hat{\mathbf{x}}_{k+1}^{-}+\mathbf{K}_{k+1}\mathbf{y}_{k+1}$'),
        (35.5,r'(6) $\mathbf{P}_{k+1}^{+}=\mathbf{P}_{k+1}^{-}-\mathbf{K}_{k+1}\mathbf{S}_{k+1}\mathbf{K}_{k+1}^{\top}$'),
    ]
    for yy,eq in correction:
        d_text(114,yy,eq,size=11.4,bounds=(110,32,194,95))

    # No: la predicción pasa a ser la salida; no se calcula innovación ni ganancia.
    d_arrow((50,56),(50,52))
    d_text(54,54,'No',size=11)
    d_box(8,32,92,52)
    d_text(50,48,'Sin corrección',size=11.5,ha='center',weight='bold',bounds=(8,32,92,52))
    d_text(50,42.5,r'$\hat{\mathbf{x}}_{k+1}^{+}=\hat{\mathbf{x}}_{k+1}^{-}$',
        size=12,ha='center',bounds=(8,32,92,52))
    d_text(50,36.4,r'$\mathbf{P}_{k+1}^{+}=\mathbf{P}_{k+1}^{-}$',
        size=12,ha='center',bounds=(8,32,92,52))

    # Ambas ramas producen una creencia en el mismo instante y alimentan el ciclo siguiente.
    d_route([(50,32),(50,16),(64,16)])
    d_route([(152,32),(152,16),(141,16)])
    d_box(64,9,141,24)
    d_text(102.5,20,r'Salida en $t_{k+1}$',size=10.5,ha='center',bounds=(64,9,141,24))
    d_text(102.5,13.8,r'$\hat{\mathbf{x}}_{k+1}^{+},\ \mathbf{P}_{k+1}^{+}$',
        size=13,ha='center',bounds=(64,9,141,24))
    d_route([(102.5,9),(102.5,4),(3,4),(3,123),(8,123)])
    d_text(40,7,r'$k\leftarrow k+1$',size=10.5,ha='center')
    d_text(100,-3.8,'EKF discreto de primer orden con ruido aditivo independiente.',size=10.5,ha='center')

    # Verificar que las ecuaciones caben en sus recuadros.
    decision_fig.canvas.draw()
    renderer=decision_fig.canvas.get_renderer()
    overflow=[]
    for artist,bounds in decision_labels:
        x0,y0,x1,y1=bounds
        a=decision_ax.transData.transform((x0,y0))
        b=decision_ax.transData.transform((x1,y1))
        extent=artist.get_window_extent(renderer)
        if extent.x0<a[0]-1 or extent.x1>b[0]+1 or extent.y0<a[1]-1 or extent.y1>b[1]+1:
            overflow.append(artist.get_text())
    if overflow:
        raise RuntimeError('Texto fuera de los recuadros: '+repr(overflow))

    decision_path=(DIAGRAM_DIR / 'ekf_general_block_diagram_paper_with_measurement.png')
    decision_fig.savefig(decision_path,dpi=300,facecolor='white')
    decision_fig.savefig(decision_path.with_suffix('.svg'),facecolor='white')
    plt.show()
    print(f'Figura simplificada con decisión guardada (PNG y SVG): {decision_path.resolve()}')


## Lectura y precisiones

1. **Inicializar:** especificar $\mathbf{f},\mathbf{h}$, la estimación $\hat{\mathbf{x}}_0^+$ y su incertidumbre $\mathbf{P}_0^+$, y las covarianzas de ruido.
2. **Predecir:** avanzar la estimación por $\mathbf{f}$ y propagar $\mathbf{P}$ mediante el Jacobiano de esa transición. Las entradas $\mathbf{u}_k$ se consideran conocidas.
3. **Comprobar si hay medición:** sin medición, conservar la predicción como salida; con medición, continuar con los siete pasos de actualización.
4. **Actualizar:** obtener la medición esperada y su Jacobiano, calcular la innovación y su covarianza, calcular la ganancia, corregir el estado y actualizar la covarianza con la forma de Joseph.
5. **Repetir:** la salida es la creencia del siguiente intervalo.

### Qué representan las matrices

- $\mathbf{P}$ aproxima la covarianza del error de estimación. Sus elementos diagonales son varianzas; los no diagonales describen covarianzas entre errores de distintos estados.
- $\mathbf{Q}_k=\operatorname{Cov}(\mathbf{w}_k)$ representa la incertidumbre de proceso acumulada durante el intervalo $[t_k,t_{k+1}]$. No se identifica automáticamente con una intensidad de ruido continuo.
- $\mathbf{R}_{k+1}=\operatorname{Cov}(\mathbf{v}_{k+1})$ representa la incertidumbre de la medición. Al cambiar los sensores disponibles se usan la medición, el modelo y la submatriz de $\mathbf{R}$ correspondientes.
- $\mathbf{S}_{k+1}$ combina la incertidumbre de la medición esperada con la del sensor. $\mathbf{K}_{k+1}$ transforma una innovación del espacio de mediciones al espacio de estados. Las covarianzas cruzadas permiten corregir estados no medidos directamente.
- $\mathbf{P}_0,\mathbf{Q},\mathbf{R}$ son simétricas semidefinidas positivas. La fórmula con $\mathbf{S}^{-1}$ requiere $\mathbf{S}$ invertible; por ejemplo, $\mathbf{R}$ definida positiva es una condición suficiente.

### Qué agrega la linealización

$$\mathbf{f}(\hat{\mathbf{x}}_k^++\delta\mathbf{x},\mathbf{u}_k)\approx\mathbf{f}(\hat{\mathbf{x}}_k^+,\mathbf{u}_k)+\mathbf{F}_k\delta\mathbf{x}.$$
$$\mathbf{h}(\hat{\mathbf{x}}_{k+1}^-+\delta\mathbf{x})\approx\mathbf{h}(\hat{\mathbf{x}}_{k+1}^-)+\mathbf{H}_{k+1}\delta\mathbf{x}.$$

Los Jacobianos pueden calcularse analíticamente, por diferenciación automática o numéricamente. Se evalúan en la estimación local; si las funciones son lineales pueden ser constantes.

### Si la dinámica original es continua

Para $\dot{\mathbf{x}}=\mathbf{g}(\mathbf{x},\mathbf{u})$, la función $\mathbf{f}$ del diagrama es el mapa entre dos instantes que se obtiene al integrar o discretizar. Su Jacobiano no es directamente $\partial\mathbf{g}/\partial\mathbf{x}$. Con Euler y entradas constantes en el intervalo:

$$\mathbf{f}(\mathbf{x},\mathbf{u})=\mathbf{x}+\Delta t_k\mathbf{g}(\mathbf{x},\mathbf{u}),\qquad
\mathbf{F}_k=\mathbf{I}+\Delta t_k\left.\frac{\partial\mathbf{g}}{\partial\mathbf{x}}\right|_{\hat{\mathbf{x}}_k^+,\mathbf{u}_k}.$$

Euler se muestra solo para explicar la relación, no como recomendación de integrador. Si se integra un modelo continuo con otro método, se debe obtener el Jacobiano de la transición correspondiente y una covarianza discreta de proceso coherente con el intervalo.

### Reducción de incertidumbre y límites

Con la ganancia del diagrama y aritmética exacta, Joseph es equivalente a
$$\mathbf{P}_{k+1}^+=\mathbf{P}_{k+1}^- -\mathbf{K}_{k+1}\mathbf{S}_{k+1}\mathbf{K}_{k+1}^{\top}\preceq\mathbf{P}_{k+1}^-.$$
La desigualdad es en el sentido semidefinido: la actualización no aumenta la varianza estimada de ninguna combinación lineal de los estados. No implica que cada elemento de la matriz disminuya ni que el error real disminuya en cada medición. Tampoco compara la salida con $\mathbf{P}_k^+$ del instante anterior. La calidad depende del modelo, de las covarianzas especificadas y de la validez de la linealización local.

La figura no cubre ruido no aditivo o correlacionado, filtros iterados ni filtros de raíz cuadrada. Si el modelo de medición depende de entradas o del tiempo, escribir $\mathbf{h}(\mathbf{x},\mathbf{u},t)$ y evaluar su Jacobiano en los valores conocidos correspondientes.

**Al implementar:** la inversa de $\mathbf{S}$ expresa la ecuación matemática; conviene resolver el sistema lineal para obtener la ganancia en vez de formar la inversa explícitamente.

Referencias: [MathWorks: algoritmo EKF y ruido aditivo/no aditivo](https://www.mathworks.com/help/control/ug/extended-and-unscented-kalman-filter-algorithms-for-online-state-estimation.html), [MathWorks: covarianzas configurables](https://www.mathworks.com/help/control/ref/extendedkalmanfilter.html), [Särkkä (2013), Bayesian Filtering and Smoothing, cap. 5](https://users.aalto.fi/~ssarkka/pub/cup_book_online_20131111.pdf#page=89).
